In [2]:
import os
from openai import OpenAI 
from dotenv import load_dotenv
from IPython.display import Markdown, display



In [3]:
load_dotenv(override=True)
openai_apikey=os.getenv('OPENAI_API_KEY')
model='gpt-5-nano'
openai=OpenAI()


In [4]:
from system import retrieve_system_info
system_info=retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '10',
  'version': '10.0.26200',
  'kernel': '10',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': 'AMD Ryzen 5 4600H with Radeon Graphics',
  'cores_logical': 12,
  'cores_physical': 6,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.exe (MinGW.org GCC-6.3.0-1) 6.3.0',
   'g++': 'g++.exe (MinGW.org GCC-6.3.0-1) 6.3.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [6]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=model, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

Short answer: You already have a C++ compiler on this system.

What you have
- Windows system with MinGW GCC in your toolchain: gcc.exe and g++.exe (MinGW.org GCC-6.3.0-1). So no installation is required to compile main.cpp.

Simple ways to compile and run from Windows

Option A: Command Prompt or PowerShell (manual steps)
- Open Command Prompt (cmd) or PowerShell.
- Change to the folder containing main.cpp:
  - cd path\to\your\folder
- Compile with optimization for fastest runtime:
  - g++.exe -O3 -std=c++17 main.cpp -o main.exe
- Run the resulting program:
  - main.exe

Option B: If you prefer a Python workflow (exact commands)

- These are the exact strings to use for compile_command and run_command:

compile_command = ["g++.exe", "-O3", "-std=c++17", "main.cpp", "-o", "main.exe"]
run_command = ["main.exe"]

- A minimal Python usage example (as you described):

import subprocess

compile_command = ["g++.exe", "-O3", "-std=c++17", "main.cpp", "-o", "main.exe"]
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)

run_command = ["main.exe"]
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)

print(run_result.stdout)

Notes and tips
- If g++.exe isn’t in your PATH, use the full path to g++.exe, e.g. "C:\\mingw-w64\\bin\\g++.exe".
- If your main.cpp relies on features not available in the older GCC (6.3.0) you have, you might need a newer MinGW-w64. In that case, you can install a newer MinGW-w64 distribution (see the optional install steps below). Otherwise, the commands above will work with the compiler you have.
- For maximum portability in Python, you might prefer ["main.exe"] as the run_command on Windows (instead of "./main.exe").

Optional: if you truly need to install a compiler (very rarely needed here)
If, for some reason, you don’t actually have g++.exe, the simplest path is MSYS2 (recommended) or MinGW-w64:

- MSYS2 approach (easy to keep up to date)
  1) Download and install MSYS2 from https://www.msys2.org
  2) Start the MSYS2 MinGW 64-bit shell (from the Start Menu)
  3) Run: pacman -Syu
  4) Close the shell, reopen, then run: pacman -S mingw-w64-x86_64-gcc
  5) Ensure C:\msys64\mingw64\bin is on your PATH (this is where g++.exe will live)
  6) Use g++.exe as shown above

- MinGW-w64 (alternative)
  1) Download the MinGW-w64 installer from the project site
  2) Install with 64-bit architecture
  3) Add the bin directory (containing g++.exe) to your PATH
  4) Use the same compile/run commands as above

If you’d like, tell me which folder your main.cpp is in and whether you’re using Command Prompt or PowerShell, and I’ll tailor the exact commands for you.

In [18]:
compile_command = ["g++.exe", "-O3", "-std=c++17", "main.cpp", "-o", "main.exe"]
run_command = ["main.exe"]



In [13]:
system_prompt=""" 
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
 """
def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [14]:
def message_for(python):
    return [
        {"role":"user" ,"content":user_prompt_for(python)},
        {"role":"system","content":system_prompt}
    ]

def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [15]:
def port(client,model,python):
    reasoning_effort = "high" if 'gpt' in model else None
    response=openai.chat.completions.create(model=model,messages=message_for(python),reasoning_effort=reasoning_effort)
    result=response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
pi= """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [16]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [11]:
run_python(pi)

Result: 3.141592656089
Execution Time: 64.640665 seconds


In [ ]:
port(openai, model, pi)